____
# COMPUTE GRADIENT, INTERPOLATE AND ROTATE SWOT DATA ON COLOC POINTS

In [62]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_2km, add_mask_inside_swot, add_grid_metrics, build_swath_polygon, coloc_swot, rotate_ggd
from diagnosis import drifters_sources

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

___________
# CHOOSE

In [149]:
# TO CHOOSE 
spectral_decomp = True
dt = '10d' #'nearestswath'
resolution = '2km'
filter_diff_method = 'diff_only'

In [150]:
preprocessed_dir = os.path.join(zarr_dir, 'before_coloc', 'preprocessed_swot',f'{resolution}_{filter_diff_method}', '*.nc')

ggd_variables = ['cvl_mean_dynamic_topography_cnes_cls_22',
                 'cvl_mean_sea_surface_cnes_22_hybrid',
                 'cvl_ocean_tide_fes_2022',
                 'cvl_ssha_reference',
                 'duacs_ssha_karin_2_calibrated',
                 'duacs_ssha_karin_2_filtered',]


variables =[#'ancillary_surface_classification_flag',
             'cross_track_distance',
             'distance_to_coast',
             'duacs_editing_flag',
             #'duacs_phase_screen',
             #'duacs_phase_screen_orbit',
             #'duacs_phase_screen_static',
             'duacs_relative_vorticity',
             'duacs_speed_meridional',
             'duacs_speed_meridional_abs',
             'duacs_speed_zonal',
             'duacs_speed_zonal_abs',
             'duacs_strain',
             'duacs_xcal',
             'sig0_karin_2',
             'phi',
             'swh_model', 
             'ssh_karin_uncert',
             'pass_number',
            ]

drifters_sources = 'all_med_variational_10min_v0.nc'
if spectral_decomp == True : drifters_sources = 'spectral_decomp_'+ drifters_sources
colocs_sources = f'{dt}_'+drifters_sources



from swot import interp_dss, rotate
def interp_coloc_one_cycle(dfr_, dsalti, cycle):#, vars_to_rotate=[]):
    # select drifter point
    dsalti_ = dsalti.sel(cycle_number=cycle)
    df_interp = interp_dss(dsalti_.drop_vars('pass_number'), dfr_.longitude.values, dfr_.latitude.values)
    df_out = pd.concat([dfr_.reset_index()[['row_number', 'longitude']].set_index('longitude'), df_interp.set_index('longitude')], axis=1).reset_index().set_index('row_number')
    #for v in vars_to_rotate :
    #    df_out[v[O]], df_out[v[1]] = rotate(df_out[v[O]], df_out[v[1]], df_out(phi))
    return df_out
    
def coloc_swot(dfr, dsalti):
    swath = dsalti.pass_number.values
    
    DF = []
    for cycle in dfr[dfr.pass_number==swath].cycle_number.unique():
        try : 
            dfr_ = dfr.where((dfr.pass_number==swath)&(dfr.cycle_number==cycle)).dropna()
            DF.append(interp_coloc_one_cycle(dfr_, dsalti, cycle))
        except : 
            print('no', cycle)
            continue
        print(cycle)
    return pd.concat(DF)


# Drifters
df = pd.read_csv(os.path.join(zarr_dir,'coloc_files', 'drifters', f'drifterscoloc_'+colocs_sources.replace('.nc', '.csv')), dtype={'drifter_id':str}, parse_dates=['datetime']).set_index('row_number')

_______
# Data

In [160]:
# Alti files
files = glob(preprocessed_dir)
files

['/Users/mdemol/DATA_MED_COLOC/before_coloc/preprocessed_swot/2km_diff_only/pass16_swot_2km_diff_only.nc',
 '/Users/mdemol/DATA_MED_COLOC/before_coloc/preprocessed_swot/2km_diff_only/pass3_swot_2km_diff_only.nc']

# Interpolate 


In [161]:
D=[]
for f in files:
    dsalti = xr.open_dataset(f).compute()
    D.append(coloc_swot(df, dsalti))
pd.concat(D, axis=0).to_csv(f.replace('before_coloc/preprocessed_swot', 'coloc_files/alti').replace('.nc', '___'+colocs_sources +'.csv'))
    

493
494
495
496
497
498
499
500
501
502
503
504
505
506
507
509
510
511
512
514
515
516
517
518
519
520
521
522
523
524
525
529
530
531
532
533
535
536
537
538
539
540
541
542
543
544
545
546
547
548
549
550
551
552
553
554
555
556
557
558
559
560
561
562
563
564
565
566
567
568
569
570
571
572
573
574
575
576
577
478
479
480
481
482
483
484
485
486
487
488
489
490
491
492
493
494
495
496
497
499
500
501
502
503
504
505
506
507
508
509
510
511
512
513
514
515
516
517
518
519
520
521
522
523
524
525
529
530
531
532
533
534
535
536
537
538
539
540
541
542
543
544
545
546
547
548
549
550
551
552
553
554
555
556
557
558
559
560
561
562
563
564
565
567
569
570
571
573
574
575
576
577
578
